<a href="https://colab.research.google.com/github/EmanuelMatiasCinel/Proyecto-Final-Data-Science/blob/main/Grupo_16_Proyecto_Data_Science_Corregido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grupo 16 - Proyecto Data Science
## Predicción de la dirección anual del precio de acciones a partir de indicadores financiero-contables


 **Objetivo**: construir un modelo de clasificación que, a partir de los indicadores financiero-contables de una empresa en el año t, prediga si su acción sube o baja durante el año t+1. Como análisis complementario, se agrupan las empresas por perfil financiero (clustering) para ver si ciertos perfiles se asocian a mayor probabilidad de suba.

**Fuente de datos**: “200+ Financial Indicators of US stocks (2014-2018)”, Kaggle, usuario Nicolas Carbone (https://www.kaggle.com/datasets/cnic92/200-financial-indicators-of-us-stocks-20142018). Cada archivo anual contiene más de 200 indicadores por empresa, el sector y la variación de precio del año siguiente.

**Variable objetivo**: `Class` (en la notebook, `clase`): 1 = la acción sube en el año siguiente, 0 = no sube. Coincide con que la variación de precio del año siguiente sea positiva.

##PARTE 1: ANÁLISIS EXPLORATORIO DE LA BASE DE DATOS

## 0. Preparación del entorno

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## 1. Carga y unificación de los 5 años

Se cargan los 5 archivos (`2014_Financial_Data.csv` a `2018_Financial_Data.csv`). Primero se mira la estructura de uno solo (2014): cuántas filas y columnas tiene y de qué tipo es cada variable.

In [ ]:
#define los data frames y lee los .csv

df= pd.read_csv("2014_Financial_Data.csv")
df1= pd.read_csv("2015_Financial_Data.csv")
df2= pd.read_csv("2016_Financial_Data.csv")
df3= pd.read_csv("2017_Financial_Data.csv")
df4= pd.read_csv("2018_Financial_Data.csv")

#imprime por consola las primeras 5 filas del DataFrame
print(df.head())

# Imprime info completa del DF
print(df.info())

# Obtengo el nombre de las columnas
df.columns

## Corrección de columnas y unificación de los 5 años

La primera columna de cada archivo, que contiene el ticker de la empresa, no tiene nombre (aparece como `Unnamed: 0`). Se renombra a `Ticker`. Además:

- Se agrega una columna `Year` con el año de cada archivo.
- La variación de precio cambia de nombre según el año (`2015 PRICE VAR [%]`, `2016 PRICE VAR [%]`, ...). Se unifica bajo el nombre `Price Var Next Year`.

Después se concatenan los 5 archivos en una sola tabla. El resto de los nombres se deja como viene en los CSV originales, porque se necesitan así para la selección de columnas del paso 3.

In [ ]:
anios = [2014, 2015, 2016, 2017, 2018]
tablas = []

for anio in anios:
    tabla = pd.read_csv(f"{anio}_Financial_Data.csv")
    tabla = tabla.rename(columns={"Unnamed: 0": "Ticker"})
    # la columna de variación de precio se llama '<anio+1> PRICE VAR [%]': la generalizamos
    columna_precio = [c for c in tabla.columns if "PRICE VAR" in c][0]
    tabla = tabla.rename(columns={columna_precio: "Price Var Next Year"})
    tabla["Year"] = anio
    tablas.append(tabla)

df_completo = pd.concat(tablas, ignore_index=True)
print("Filas:", df_completo.shape[0], "| Columnas:", df_completo.shape[1])
df_completo.head()

In [ ]:
#comprobación

print("Dimensiones:", df_completo.shape)


print("\nRegistros por año:\n", df_completo["Year"].value_counts())

print("\nDuplicados por Ticker y Año:", df_completo.duplicated(subset=["Ticker", "Year"]).sum())


print("\nSectores únicos:\n", df_completo["Sector"].unique())

In [ ]:
#guardamos el dataset unificado
df_completo.to_csv("financial_data_unificado.csv", index=False)

## 2. Diagnóstico inicial (sobre el dataset completo)

Las tres preguntas de siempre: cuántas filas y columnas, de qué tipo es cada variable, y qué tan completa está. Lo hacemos sobre las 225 columnas originales, antes de recortar, para tener una foto completa del dataset crudo.

In [ ]:
# Obtengo el nombre de las columnas
df_completo.info()

In [ ]:
# Tipo de datos en cada columna
df_completo.dtypes

### Faltantes en el dataset completo

In [ ]:
# Porcentaje de faltantes por columna (las 10 con más faltantes)
nulos_columna = (df_completo.isnull().mean() * 100).sort_values(ascending=False).round(2)
print("Columnas con más faltantes (% de filas):")
print(nulos_columna.head(10))

Hay columnas casi vacías: la peor tiene 99,9% de faltantes. Por eso no conviene trabajar con las 225 columnas, y en la sección 3 se hace una selección por criterio financiero.

### Duplicados y empresas

In [ ]:
#Analizamos la existencia de filas Duplicadas
print("Filas duplicadas:", df_completo.duplicated().sum())
print("Empresas (tickers) distintas:", df_completo["Ticker"].nunique())
print("Sectores:", df_completo["Sector"].nunique())
df_completo["Sector"].value_counts()

No hay filas duplicadas ni repeticiones de empresa dentro de un mismo año. El dataset tiene 4.980 empresas distintas y 11 sectores.

### Sectores

In [ ]:
#Contamos la cantidad de empresas por sector y año
tabla = df_completo.groupby(['Sector', 'Year']).size().unstack(fill_value=0)
display(tabla)

Financial Services es el sector con más empresas (4.720 filas en total) y su tamaño no es estable: pasa de 660 empresas en 2014 a 1.247 en 2017 y baja a 824 en 2018. Esto importa más adelante, porque es el sector con más faltantes.

In [ ]:
#Columnas
print("lista de columnas",df_completo.columns.tolist())

# Obtengo el promedio de la columna (Revenue-Ingresos Totales-permite calcular cuál fue la facturación promedio de las empresas por año.)

print("La facturación promedio de las empresas por año fue de: ",df_completo.groupby('Year' )["Revenue"].mean())

# Obtengo el promedio por sector y año (.unstack() para que se vizualizar bien la tabla de doble entrada y .display() genera una tabla HTML interactiva)

print("--- Facturación promedio por sector y año ---")
display(df_completo.groupby(['Sector', 'Year'])['Revenue'].mean().unstack())

## 3. Selección de columnas

Del dataset completo (225 columnas) nos quedamos con las que definimos por criterio financiero: valuación, rentabilidad, liquidez, endeudamiento, crecimiento y dividendos, más `Ticker`, `Year` y `Sector` para identificar cada fila, y la variable objetivo (`Price Var Next Year` y `Class`).

##De la base de datos completa se utilizaran solamente las siguientes columnas:
**Valuación — "¿está cara o barata la acción?"**

•	**P/E (Price/Earnings):** cuántos años de ganancia actual hacen falta para "pagar" el precio de la acción. Bajo = parece barata (o el mercado no confía en su futuro); alto = cara (o el mercado espera que crezca mucho).

•	**P/B (Price/Book):** precio de la acción vs. el valor contable de sus activos. Sirve para ver si el mercado paga mucho más de lo que la empresa "vale en el papel".

**Rentabilidad — "¿gana plata de forma eficiente?"**

•	**ROE (Return on Equity):** por cada $1 que pusieron los accionistas, cuánta ganancia generó la empresa. Es el ratio más citado en análisis fundamental.

•	**ROA (Return on Assets):** El ROA mide cuánta ganancia genera una empresa por cada peso (o dólar) que tiene invertido en sus activos totales

•	**ROIC (Return of Invested Capital):** El ROIC mide cuánta ganancia operativa genera una empresa por cada peso (o dólar) de capital que realmente se usó para financiar el negocio

•	**Net Profit Margin:** de cada $100 que factura, cuántos terminan siendo ganancia neta (después de todos los gastos e impuestos).

**Liquidez — "¿puede pagar sus deudas de corto plazo?"**

•	**Current Ratio:** activos que puede convertir en efectivo rápido, divididos por lo que debe pagar pronto. Menor a 1 es señal de alerta (podría no cubrir sus obligaciones inmediatas).

•	**Quick Ratio:** indicador financiero que mide la capacidad de una empresa para pagar sus deudas a corto plazo usando solo sus activos más líquidos. Es más estricto y conservador porque excluye los inventarios, ya que estos tardan más tiempo en convertirse en dinero en efectivo.

•	**Free Cash Flow Margin:** % de las ventas que se convierte en caja libre (comparable entre empresas)

**Endeudamiento — "¿cuánta deuda tiene en relación a lo suyo?"**

•	**Debt to Equity:** deuda total / patrimonio de los accionistas. Alto = la empresa se financia mucho con deuda (más riesgo si suben las tasas o caen las ventas).

• **Interest Coverage:** Mide cuántas veces la ganancia operativa cubre los intereses de la deuda.

**Crecimiento — "¿está creciendo o estancada?"**

•	**Revenue Growth (YoY):** cuánto crecieron las ventas respecto al año anterior. Un solo número, no una serie — está permitido.

•	**EPS Growth (YoY):** cuánto creció la ganancia por acción respecto al año anterior.

**Dividendos y contexto**

•	**Dividend Yield:** qué % del precio de la acción se devuelve como dividendo anual.

•	**Market Cap:** tamaño de la empresa (grande/mediana/chica) — importante porque empresas grandes y chicas se comportan distinto.

•	**Sector:** variable categórica que permite comparar empresas de la misma industria y ver si el comportamiento cambia según el sector.

•	**Variable objetivo:** `Price Var Next Year` (variación del precio de la acción en el año siguiente) y `Class` (1 si esa variación es positiva, 0 si no). No se usan como predictoras: son lo que el modelo tiene que predecir.

In [ ]:
columnas_conservar = [
    # Identificadores y objetivo
    "Ticker", "Year", "Sector", "Market Cap", "Price Var Next Year", "Class",
    # Valoración
    "PE ratio", "PB ratio",
    # Rentabilidad
    "ROE", "returnOnAssets", "ROIC", "Net Profit Margin", "Free Cash Flow margin",
    # Liquidez
    "Current ratio", "quickRatio",
    # Apalancamiento
    "Debt to Equity", "Interest Coverage",
    # Crecimiento
    "Revenue Growth", "EPS Growth",
    # Dividendos
    "Dividend Yield",
]
# Nos quedamos solo con las columnas de columnas_conservar
df_reducido = df_completo[columnas_conservar].copy()
print("Columnas:", df_reducido.shape[1], "| Filas:", df_reducido.shape[0])
df_reducido.head()

### Traducción de los nombres de las columnas

Con el dataset ya recortado, se renombran las columnas al español (en minúscula, sin espacios y con guion bajo), manteniendo las siglas del inglés que también se usan en español (`pe_ratio`, `roe`, `roa`, `roic`). La columna `Class` pasa a llamarse `clase`.

In [ ]:
traduccion = {
    "Ticker": "ticker",
    "Year": "anio",
    "Sector": "sector",
    "Market Cap": "capitalizacion_mercado",
    "PE ratio": "pe_ratio",
    "PB ratio": "pb_ratio",
    "ROE": "roe",
    "returnOnAssets": "roa",
    "ROIC": "roic",
    "Net Profit Margin": "margen_neto",
    "Free Cash Flow margin": "margen_flujo_caja_libre",
    "Current ratio": "liquidez_corriente",
    "quickRatio": "liquidez_inmediata",
    "Debt to Equity": "deuda_patrimonio",
    "Interest Coverage": "cobertura_intereses",
    "Revenue Growth": "crecimiento_ingresos",
    "EPS Growth": "crecimiento_eps",
    "Dividend Yield": "rendimiento_dividendo",
    "Price Var Next Year": "variacion_precio_siguiente_anio",
    "Class": "clase",
}

df_reducido = df_reducido.rename(columns=traduccion)
print(df_reducido.columns.tolist())

### Ceros que son valores faltantes

En `pe_ratio` y `pb_ratio` un valor igual a 0 no es un dato real: indica que el dato no existe (por ejemplo, ganancias o patrimonio no positivos; el dataset no lo aclara). Se reemplazan por faltantes. Son 6.852 ceros en `pe_ratio` y 1.046 en `pb_ratio`.

In [ ]:
# En pe_ratio y pb_ratio un 0 no es un valor real: indica dato ausente
cols_cero_es_nan = ["pe_ratio", "pb_ratio"]
print("Ceros antes:", (df_reducido[cols_cero_es_nan] == 0).sum().to_dict())
df_reducido[cols_cero_es_nan] = df_reducido[cols_cero_es_nan].replace(0, np.nan)
print("Ceros después:", (df_reducido[cols_cero_es_nan] == 0).sum().to_dict())

## 4. Valores faltantes


In [ ]:
faltantes = pd.DataFrame({
    "cantidad": df_reducido.isna().sum(),
    "porcentaje": (df_reducido.isna().mean() * 100).round(2),
})
faltantes[faltantes["cantidad"] > 0].sort_values("porcentaje", ascending=False)

Con los ceros tratados como faltantes, `pe_ratio` es la columna con más datos vacíos (41,2%), seguida por `roa`, `liquidez_corriente` y `roic` (cerca del 30% cada una) y `pb_ratio` (27,3%).

In [ ]:
# Lista de los 15 ratios financieros (excluye identificadores y columnas de validación)
columnas_ratio = [
    "capitalizacion_mercado", "pe_ratio", "pb_ratio", "roe", "roa", "roic",
    "margen_neto", "margen_flujo_caja_libre", "liquidez_corriente", "liquidez_inmediata",
    "deuda_patrimonio", "cobertura_intereses", "crecimiento_ingresos",
    "crecimiento_eps", "rendimiento_dividendo",
]

# Cuántos de esos 15 ratios le faltan a cada fila (empresa-año)
df_reducido["faltantes_en_fila"] = df_reducido[columnas_ratio].isnull().sum(axis=1)

# Distribución: ¿cuántas filas tienen 0 faltantes, 1 faltante, ..., 15 faltantes?
print("=== Distribución de faltantes por fila ===")
print(df_reducido["faltantes_en_fila"].value_counts().sort_index())

La mayoría de las filas tiene pocos ratios vacíos, pero hay un grupo de 2.300 filas con 9 o más de los 15 ratios sin dato, entre ellas 1.199 con los 15 vacíos.

## ¿Es aleatorio el faltante o se concentra en algún sector?

In [ ]:
print("\n=== Promedio de faltantes por sector ===")
print(df_reducido.groupby("sector")["faltantes_en_fila"].mean().sort_values(ascending=False))

El faltante no es aleatorio: depende del sector. Financial Services tiene en promedio 6,7 de los 15 ratios vacíos por fila, contra valores de entre 1,0 y 2,0 en casi todos los demás sectores. Real Estate (3,9) y Communication Services (3,0) también quedan por encima. Es esperable, porque ratios como la liquidez o el ROIC no aplican a bancos y aseguradoras.

## ¿Se concentra en algún año?

In [ ]:
print("\n=== Promedio de faltantes por año ===")
print(df_reducido.groupby("anio")["faltantes_en_fila"].mean())

2016 y 2017 tienen más faltantes (3,3 y 3,4 ratios por fila) que 2014, 2015 y 2018 (entre 1,9 y 2,2). Coincide con los años en que Financial Services pasa de 769 a más de 1.200 empresas, lo que probablemente lo explique.

## ¿Hay empresas con todos los ratios vacíos en todos sus años?

In [ ]:
faltantes_por_ticker = df_reducido.groupby("ticker")["faltantes_en_fila"].mean()
tickers_vacios = faltantes_por_ticker[faltantes_por_ticker == 15]
print(f"Empresas con los 15 ratios vacíos en todos sus años: {len(tickers_vacios)}")

483 de las 4.980 empresas tienen los 15 ratios vacíos en todos sus años. No aportan información y salen en el filtro de filas.

## Conclusión

- Los ceros de `pe_ratio` y `pb_ratio` no son valores reales, así que se tratan como faltantes. Consecuencia: `pe_ratio` queda con 41,2% de datos vacíos.
- Se eliminan las filas con 9 o más de los 15 ratios vacíos (2.300 filas, 10,4% del total): con más de la mitad de la fila sin datos, imputarla equivaldría a inventarla. La eliminación se concentra en Financial Services (1.467 de las 2.300 filas).
- El resto de los faltantes se completa con la mediana del grupo sector + año.
- Para el modelo conviene agregar un indicador de "dato faltante" por ratio, porque que falte el P/E puede ser informativo.

In [ ]:
# Guardamos cuántas filas había antes, para poder reportar el impacto
filas_antes = len(df_reducido)

# Eliminamos las filas con 9 o más de los 15 ratios vacíos (más de la mitad de la fila)
df_reducido = df_reducido[df_reducido["faltantes_en_fila"] < 9].copy()

filas_despues = len(df_reducido)

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Filas eliminadas:", filas_antes - filas_despues,
      f"({(filas_antes - filas_despues) / filas_antes * 100:.1f}% del total)")

In [ ]:
#Completa el resto de los faltantes con la mediana del grupo sector + anio
# Imputación por grupo: mediana dentro de cada combinación sector-año
for col in columnas_ratio:
    df_reducido[col] = df_reducido.groupby(["sector", "anio"])[col].transform(
        lambda serie: serie.fillna(serie.median())
    )

print("Faltantes restantes:", int(df_reducido[columnas_ratio].isnull().sum().sum()))
df_reducido.isnull().sum()

Con esta imputación, cada empresa que tenía un ratio faltante recibe el valor típico de su sector y año en ese ratio. Es un supuesto metodológico que hay que mencionar en el informe final: reduce la variabilidad y puede suavizar diferencias reales, sobre todo en `pe_ratio` (41,2% imputado) y en Financial Services.

In [ ]:
# columna auxiliar: no puede entrar al modelo
df_reducido = df_reducido.drop(columns="faltantes_en_fila")

## 5. Outliers


In [ ]:
ncols = 4
nrows = -(-len(columnas_ratio) // ncols)  # redondeo hacia arriba
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))

for eje, columna in zip(axes.flat, columnas_ratio):
    sns.boxplot(y=df_reducido[columna], ax=eje, color="#4C72B0")
    eje.set_title(columna)
    eje.set_ylabel("")

for eje in axes.flat[len(columnas_ratio):]:
    eje.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def winsorizar(serie, percentil_bajo=0.01, percentil_alto=0.99):
    """Recorta una serie a sus percentiles bajo y alto."""
    limite_bajo = serie.quantile(percentil_bajo)
    limite_alto = serie.quantile(percentil_alto)
    return serie.clip(limite_bajo, limite_alto)

In [ ]:
# Copia winsorizada (se conserva df_reducido para comparar)
df_winsor = df_reducido.copy()
for col in columnas_ratio:
    df_winsor[col] = winsorizar(df_winsor[col])

# Antes y después, en las variables con valores más extremos
variables = ["cobertura_intereses", "margen_neto", "pe_ratio", "crecimiento_eps"]
fig, ejes = plt.subplots(2, len(variables), figsize=(16, 7))
for j, v in enumerate(variables):
    sns.boxplot(y=df_reducido[v], ax=ejes[0, j], color="#4C72B0")
    ejes[0, j].set_title(f"{v} (original)")
    sns.boxplot(y=df_winsor[v], ax=ejes[1, j], color="#55A868")
    ejes[1, j].set_title(f"{v} (winsorizado)")
    for fila in (0, 1):
        ejes[fila, j].set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
resumen_outliers = pd.DataFrame({
    "min_original": df_reducido[columnas_ratio].min(),
    "max_original": df_reducido[columnas_ratio].max(),
    "min_winsor": df_winsor[columnas_ratio].min(),
    "max_winsor": df_winsor[columnas_ratio].max(),
}).round(2)
resumen_outliers

Los ratios tienen valores extremos muy lejos del resto: por ejemplo, `roe` va de -34.772 a +11.141.100 y `cobertura_intereses` de -223.603 a +361.207, cuando la gran mayoría de las empresas se mueve en rangos de pocas unidades. Con esos valores los boxplots quedan aplastados y los modelos lineales se distorsionan.

Se recorta cada ratio en los percentiles 1 y 99 (winsorización): se conservan todas las filas, pero los extremos pasan a valer el límite (por ejemplo, `roe` queda entre -4,73 y 4,94). No se recortan la variable objetivo ni `clase`. Para el modelo, estos límites se calcularán solo con los datos de entrenamiento.

In [ ]:
df_winsor.to_csv("financial_data_limpio.csv", index=False)

## 6. Variable objetivo

In [ ]:
tabla_clase = df_reducido.groupby("anio")["clase"].agg(proporcion_sube="mean", empresas="size").round(3)
display(tabla_clase)
print("Proporción general que sube:", round(df_reducido["clase"].mean(), 3))
coincide = ((df_reducido["variacion_precio_siguiente_anio"] > 0).astype(int) == df_reducido["clase"]).mean()
print(f"clase coincide con (variación > 0) en el {coincide:.1%} de las filas")

fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
tabla_clase["proporcion_sube"].plot(kind="bar", ax=ejes[0], color="#4C72B0", rot=0)
ejes[0].axhline(df_reducido["clase"].mean(), color="gray", linestyle="--", label="promedio general")
ejes[0].set_title("Proporción de acciones que suben, por año")
ejes[0].set_xlabel("año de análisis")
ejes[0].legend()
sns.histplot(df_reducido["variacion_precio_siguiente_anio"].clip(-100, 200), bins=60, ax=ejes[1], color="#55A868")
ejes[1].set_title("Variación de precio del año siguiente (recortada a -100%/200%)")
ejes[1].set_xlabel("variación %")
plt.tight_layout()
plt.show()
display(df_reducido["variacion_precio_siguiente_anio"].describe(percentiles=[.01, .5, .99]).round(1))

In [ ]:
tabla_sector = df_reducido.pivot_table(index="sector", columns="anio", values="clase", aggfunc="mean")
plt.figure(figsize=(8, 5))
sns.heatmap(tabla_sector, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1)
plt.title("Proporción de acciones que suben, por sector y año")
plt.tight_layout()
plt.show()

- La proporción de acciones que suben cambia mucho según el año: 42,7% (2014), 69,9% (2015), 63,9% (2016), 28,2% (2017) y 68,2% (2018). Eso refleja el mercado en general más que a cada empresa.
- Los sectores siguen casi el mismo patrón: 2017 bajo y 2015 alto en la mayoría. Hay excepciones, como Healthcare en 2015 (35%) y Utilities en 2017 (56%).
- Consecuencia para el modelo: la proporción de clases del test depende del año elegido, así que hay que comparar contra un piso y mirar AUC y F1, no solo la exactitud.
- La variación de precio tiene valores extremos (máximo de +2.418.601%, con 38 filas por encima del +1.000%). `clase` coincide con "variación > 0" en el 99,9% de las filas y es una variable más robusta para clasificar.

## 7. Ratios y variable objetivo

In [ ]:
variables = ["roe", "roic", "margen_neto", "margen_flujo_caja_libre", "cobertura_intereses", "deuda_patrimonio"]
fig, ejes = plt.subplots(2, 3, figsize=(14, 7))
for eje, v in zip(ejes.flat, variables):
    sns.boxplot(data=df_winsor, x="clase", y=v, ax=eje, showfliers=False, palette=["#C44E52", "#55A868"])
    eje.set_title(v)
    eje.set_xlabel("clase (0 = no sube, 1 = sube)")
    eje.set_ylabel("")
plt.tight_layout()
plt.show()
display(df_winsor.groupby("clase")[columnas_ratio].median().T.round(3))

In [ ]:
corr_ratios = df_winsor[columnas_ratio].corr(method="spearman")
plt.figure(figsize=(10, 8))
sns.heatmap(corr_ratios, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, annot_kws={"size": 7})
plt.title("Correlación de Spearman entre ratios")
plt.tight_layout()
plt.show()

corr_clase = df_winsor[columnas_ratio].corrwith(df_winsor["clase"], method="spearman").sort_values()
plt.figure(figsize=(8, 5))
corr_clase.plot(kind="barh", color=["#C44E52" if v < 0 else "#55A868" for v in corr_clase])
plt.axvline(0, color="gray", linewidth=1)
plt.xlabel("Correlación de Spearman con clase")
plt.title("Asociación de cada ratio con que la acción suba")
plt.tight_layout()
plt.show()

- Las empresas cuya acción sube tienen, en mediana, mayor rentabilidad: `roa` 0,100 contra 0,046, `roic` 0,062 contra 0,035, `margen_neto` 0,061 contra 0,022 y `margen_flujo_caja_libre` 0,063 contra 0,021. También tienen mayor cobertura de intereses (1,33 contra 0,12) y mayor capitalización.
- Las asociaciones individuales son débiles: la correlación de Spearman más alta es 0,17 (`margen_neto`), y `crecimiento_ingresos` y `crecimiento_eps` no se asocian prácticamente con la clase. Ningún ratio por sí solo separa bien las clases, así que se espera un poder predictivo modesto.
- Hay ratios muy relacionados entre sí: `roa`–`roic` (0,69), `liquidez_corriente`–`liquidez_inmediata` (0,67) y `roic`–`margen_neto` (0,66). Esto es multicolinealidad: afecta a los modelos lineales y los árboles lo toleran mejor.

## 8. Estructura temporal y por empresa

In [ ]:
display(df_winsor.groupby("anio")[columnas_ratio].median().T.round(3))

anios_por_ticker = df_reducido.groupby("ticker")["anio"].nunique().value_counts().sort_index()
print("Empresas distintas:", df_reducido["ticker"].nunique())
anios_por_ticker.plot(kind="bar", color="#4C72B0", rot=0)
plt.xlabel("cantidad de años en que aparece la empresa")
plt.ylabel("cantidad de empresas")
plt.title("Cuántos años aparece cada empresa")
plt.tight_layout()
plt.show()

- Algunos ratios cambian de un año a otro: la mediana de `crecimiento_eps` es 0,195 en 2018 contra 0,045-0,104 entre 2014 y 2017, y la de `cobertura_intereses` baja de 1,35 (2014) a 0,40 (2016). Esto importa porque el modelo se entrena con unos años y se evalúa en otro.
- De las 4.152 empresas, 3.555 (85,6%) aparecen en los 5 años. Por eso la validación cruzada tiene que agrupar por empresa, para que una misma empresa no quede a la vez en entrenamiento y en prueba.

## 9. Conclusiones del EDA

1. **Datos:** 22.077 filas (empresa-año), de las que quedan 19.777 después de la limpieza. Son 4.152 empresas, 11 sectores y 15 ratios.
2. **Faltantes:** los ceros de `pe_ratio` y `pb_ratio` se trataron como faltantes (`pe_ratio` queda con 41,2% de datos vacíos), se eliminaron las filas con 9 o más ratios vacíos y el resto se imputó con la mediana de su sector y año.
3. **Outliers:** los ratios se recortaron en los percentiles 1 y 99.
4. **Variable objetivo:** la proporción de acciones que suben depende mucho del año (28% a 70%). Por eso se parte por año (por ejemplo, entrenar 2014-2017 y evaluar 2018) y se mide con AUC y F1 además de la exactitud.
5. **Señal de los ratios:** las asociaciones con la clase son débiles (correlación de Spearman máxima de 0,17), con la rentabilidad como la familia más asociada. Se espera un poder predictivo modesto.
6. **Para el modelo:** agrupar por empresa en la validación cruzada, mantener la imputación y el recorte dentro del `Pipeline` (ajustados solo con el train), agregar un indicador de dato faltante por ratio, usar `capitalizacion_mercado` en escala logarítmica y comparar contra el objetivo alternativo "sube más que la mediana de su año".

## Parte 2: ARMADO DE MODELOS

## 0. Preparación del entorno

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, balanced_accuracy_score,
                             classification_report, f1_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
SEMILLA = 42

## 1. Datos: se repiten las decisiones del EDA

Partimos del archivo unificado que guardó el EDA (`financial_data_unificado.csv`) y repetimos, en una sola celda, las decisiones ya justificadas allí:

1. Nos quedamos con las 15 variables elegidas, más `Ticker`, `Year`, `Sector` y el objetivo, y las traducimos.
2. Los ceros de `pe_ratio` y `pb_ratio` se tratan como faltantes.
3. Se eliminan las filas con 9 o más de los 15 ratios vacíos.

Además se anulan 15 capitalizaciones iguales a 0 y 5 liquidez corriente negativas, valores imposibles.